In [1]:
import json, os
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
transcripts_dir = root / "data" / "transcripts"
alignments_dir  = root / "data" / "alignments"

print(len(list(transcripts_dir.glob("*.json"))), "transcripts |",
      len(list(alignments_dir.glob("*.json"))), "alignments")

31 transcripts | 31 alignments


In [2]:
def joined(transcript_path):
    """The transcript with the alignment's words added. Raises ValueError if the two don't match."""
    t = json.loads(transcript_path.read_text(encoding="utf-8"))
    a = json.loads((alignments_dir / transcript_path.name).read_text(encoding="utf-8"))

    if a["audio"] != t["audio"]:
        raise ValueError(f"different audio: {t['audio']!r} vs {a['audio']!r}")

    text_words = t["text"].split()
    aligned_words = [w["word"] for w in a["words"]]
    if aligned_words != text_words:
        i = next((i for i, (x, y) in enumerate(zip(text_words, aligned_words)) if x != y),
                 min(len(text_words), len(aligned_words)))
        raise ValueError(f"words differ from word {i} ({len(text_words)} in text, {len(aligned_words)} aligned): "
                         f"text {text_words[i:i + 3]} vs alignment {aligned_words[i:i + 3]}")

    record = {k: v for k, v in t.items() if k != "words"}
    record["words"] = a["words"]
    return record


def write_json(path, record):
    """Pretty JSON with one word per line; atomic, so an interrupted run can't leave a half-written file."""
    head = {k: v for k, v in record.items() if k != "words"}
    lines = [f"    {json.dumps(k)}: {json.dumps(v, ensure_ascii=False)}" for k, v in head.items()]
    words = ",\n".join("        " + json.dumps(w, ensure_ascii=False) for w in record["words"])
    lines.append(f'    "words": [\n{words}\n    ]')
    tmp = path.with_suffix(".json.tmp")
    tmp.write_text("{\n" + ",\n".join(lines) + "\n}\n", encoding="utf-8")
    os.replace(tmp, path)


def run(write):
    ok, no_alignment, mismatched = [], [], []
    for path in sorted(transcripts_dir.glob("*.json")):
        if not (alignments_dir / path.name).exists():
            no_alignment.append(path.name)
            continue
        try:
            record = joined(path)
        except ValueError as e:
            mismatched.append(path.name)
            print(f"SKIP {path.stem[:60]}: {e}")
            continue
        if write:
            write_json(path, record)
        ok.append(path.name)
    print(f"{'joined' if write else 'ready to join'}: {len(ok)} | "
          f"no alignment: {len(no_alignment)} | mismatched: {len(mismatched)}")
    for name in no_alignment:
        print("  no alignment:", name)
    return ok, no_alignment, mismatched

## Check (writes nothing)

In [3]:
_ = run(write=False)

ready to join: 31 | no alignment: 0 | mismatched: 0


## Join (rewrites `data/transcripts/*.json`)

In [4]:
_ = run(write=True)

joined: 31 | no alignment: 0 | mismatched: 0


## Look at one

In [5]:
t = json.loads((transcripts_dir / "abuser.json").read_text(encoding="utf-8"))
print(list(t), "|", len(t["words"]), "words")
for w in t["words"][:10]:
    print(f"{w['start']:7.2f} → {w['end']:7.2f}  {w['word']}")

['audio', 'language', 'language_probability', 'duration', 'text', 'words'] | 186 words
   0.00 →    0.64  O'zbekistonda
   0.72 →    1.12  Naviyni
   1.20 →    1.52  yutgan
   1.76 →    1.84  odamlar
   1.92 →    2.16  borligini
   2.24 →    3.04  bilarmidinglar?
   3.12 →    3.20  Bir
   3.36 →    3.92  oltidan.
   3.92 →    4.24  Yo'q,
   4.24 →    4.56  yo'q,
